<a href="https://colab.research.google.com/github/Fukimodoshi/Fukimodoshi/blob/main/Forecast%20World%20Cup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# BLOQUE 1: Descargar datos de partidos internacionales
import pandas as pd

url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
df = pd.read_csv(url)

print(f"Total de partidos descargados: {len(df)}")
print(f"\nColumnas disponibles:")
print(df.columns.tolist())
print(f"\nPrimeros 3 registros:")
df.head(3)

# BLOQUE 2: Filtrar solo Mundiales y Clasificatorias desde 1990
torneos_validos = [
    "FIFA World Cup",
    "FIFA World Cup qualification"
]

df['date'] = pd.to_datetime(df['date'])

df_filtrado = df[
    (df['tournament'].isin(torneos_validos)) &
    (df['date'].dt.year >= 1990)
].copy()

print(f"Partidos de Mundial y Clasificatorias desde 1990: {len(df_filtrado)}")
print(f"\nTorneos incluidos:")
print(df_filtrado['tournament'].value_counts())

# BLOQUE 3: Convertir partidos a formato "por equipo"
# Cada partido genera DOS filas: una por cada equipo

partidos_local = df_filtrado[['date','home_team','away_team','home_score','away_score']].copy()
partidos_local.columns = ['date','equipo','rival','goles_favor','goles_contra']

partidos_visitante = df_filtrado[['date','away_team','home_team','away_score','home_score']].copy()
partidos_visitante.columns = ['date','equipo','rival','goles_favor','goles_contra']

partidos = pd.concat([partidos_local, partidos_visitante], ignore_index=True)

print(f"Total de registros por equipo: {len(partidos)}")
print(f"\nEjemplo — partidos de Brasil:")
partidos[partidos['equipo'] == 'Brazil'].head(5)

# BLOQUE 4: Calcular fuerza de ataque y defensa por equipo
fuerza = partidos.groupby('equipo').agg(
    partidos_jugados   = ('goles_favor',  'count'),
    goles_marcados     = ('goles_favor',  'sum'),
    goles_recibidos    = ('goles_contra', 'sum')
).reset_index()

# Promedios por partido
fuerza['ataque']  = fuerza['goles_marcados']  / fuerza['partidos_jugados']
fuerza['defensa'] = fuerza['goles_recibidos'] / fuerza['partidos_jugados']

# Promedio global (referencia del modelo)
promedio_goles = partidos['goles_favor'].mean()
print(f"Promedio global de goles por partido: {promedio_goles:.3f}")

# Índices normalizados respecto al promedio global
fuerza['idx_ataque']  = fuerza['ataque']  / promedio_goles
fuerza['idx_defensa'] = fuerza['defensa'] / promedio_goles

fuerza = fuerza.sort_values('idx_ataque', ascending=False).reset_index(drop=True)

print(f"\nTop 10 equipos más goleadores:")
print(fuerza[['equipo','partidos_jugados','ataque','defensa','idx_ataque','idx_defensa']].head(10).to_string(index=False))

# BLOQUE 5: Instalar librería estadística (solo la primera vez)
!pip install scipy --quiet

from scipy.stats import poisson
import numpy as np

print("Librerías listas ✓")

# BLOQUE 6: Función de predicción de partido
# La lógica: goles_esperados = idx_ataque_equipo × idx_defensa_rival × promedio_global

def predecir_partido(equipo_a, equipo_b, n_simulaciones=10000):
    """
    Predice el resultado entre dos equipos.
    Devuelve goles esperados y probabilidades de victoria/empate/derrota.
    """
    # Buscar índices de cada equipo
    fila_a = fuerza[fuerza['equipo'] == equipo_a]
    fila_b = fuerza[fuerza['equipo'] == equipo_b]

    if fila_a.empty:
        print(f"Equipo no encontrado: {equipo_a}")
        return
    if fila_b.empty:
        print(f"Equipo no encontrado: {equipo_b}")
        return

    idx_atq_a  = fila_a['idx_ataque'].values[0]
    idx_def_a  = fila_a['idx_defensa'].values[0]
    idx_atq_b  = fila_b['idx_ataque'].values[0]
    idx_def_b  = fila_b['idx_defensa'].values[0]

    # Goles esperados por cada equipo
    lambda_a = idx_atq_a * idx_def_b * promedio_goles
    lambda_b = idx_atq_b * idx_def_a * promedio_goles

    # Simulación Monte Carlo: 10,000 partidos simulados
    goles_a = poisson.rvs(lambda_a, size=n_simulaciones)
    goles_b = poisson.rvs(lambda_b, size=n_simulaciones)

    # Probabilidades
    prob_a    = np.mean(goles_a > goles_b)
    prob_emp  = np.mean(goles_a == goles_b)
    prob_b    = np.mean(goles_a < goles_b)

    print(f"\n{'='*45}")
    print(f"  {equipo_a}  vs  {equipo_b}")
    print(f"{'='*45}")
    print(f"  Goles esperados:  {lambda_a:.2f}  —  {lambda_b:.2f}")
    print(f"{'─'*45}")
    print(f"  Victoria {equipo_a:<20} {prob_a*100:.1f}%")
    print(f"  Empate                         {prob_emp*100:.1f}%")
    print(f"  Victoria {equipo_b:<20} {prob_b*100:.1f}%")
    print(f"{'='*45}")

    return lambda_a, lambda_b, prob_a, prob_emp, prob_b

# Prueba con un partido clásico
predecir_partido("Brazil", "Argentina")

# BLOQUE 7: Lista de los 48 equipos clasificados al Mundial 2026
equipos_mundial = [
    # CONMEBOL
    "Brazil", "Argentina", "Uruguay", "Colombia", "Ecuador", "Paraguay", "Venezuela", "Bolivia",
    # UEFA
    "Spain", "France", "England", "Germany", "Portugal", "Netherlands", "Belgium",
    "Italy", "Croatia", "Switzerland", "Austria", "Scotland", "Norway", "Sweden",
    "Turkey", "Czechia", "Bosnia and Herzegovina",
    # CONCACAF
    "United States", "Mexico", "Canada", "Costa Rica", "Honduras", "Jamaica", "Panama",
    # CAF
    "Morocco", "Senegal", "Egypt", "Ivory Coast", "Algeria", "Ghana",
    "South Africa", "Tunisia", "Cape Verde", "DR Congo",
    # AFC
    "Japan", "South Korea", "Saudi Arabia", "Australia", "Iran",
    "Iraq", "Jordan", "Qatar", "Uzbekistan",
    # OFC
    "New Zealand"
    ]

# Verificar cuáles están en nuestro dataset
equipos_encontrados = []
equipos_faltantes   = []

for eq in equipos_mundial:
    if eq in fuerza['equipo'].values:
        equipos_encontrados.append(eq)
    else:
        equipos_faltantes.append(eq)

print(f"Equipos encontrados en el dataset: {len(equipos_encontrados)}")
print(f"Equipos NO encontrados:            {len(equipos_faltantes)}")
if equipos_faltantes:
    print(f"\nFaltantes: {equipos_faltantes}")

    # BLOQUE 8: Tabla de ranking — fuerza ofensiva de los 48 equipos
fuerza_mundial = fuerza[fuerza['equipo'].isin(equipos_encontrados)].copy()
fuerza_mundial = fuerza_mundial.sort_values('idx_ataque', ascending=False).reset_index(drop=True)
fuerza_mundial.index += 1  # ranking desde 1

print("RANKING OFENSIVO — MUNDIAL 2026")
print(f"{'#':<4} {'Equipo':<25} {'Ataque':>8} {'Defensa':>8} {'Idx Ataque':>11} {'Idx Defensa':>12}")
print("─" * 72)
for i, row in fuerza_mundial.iterrows():
    print(f"{i:<4} {row['equipo']:<25} {row['ataque']:>8.2f} {row['defensa']:>8.2f} {row['idx_ataque']:>11.3f} {row['idx_defensa']:>12.3f}")

# BLOQUE 9: Corregir nombre de Czechia y verificar
# En el dataset histórico figura como "Czech Republic"

equipos_mundial_corregido = [eq if eq != "Czechia" else "Czech Republic" for eq in equipos_mundial]

# Reverificar
equipos_encontrados = []
equipos_faltantes   = []

for eq in equipos_mundial_corregido:
    if eq in fuerza['equipo'].values:
        equipos_encontrados.append(eq)
    else:
        equipos_faltantes.append(eq)

print(f"Equipos encontrados: {len(equipos_encontrados)}/48")
print(f"Faltantes: {equipos_faltantes}")

# Actualizar tabla de fuerza con los 48 equipos
fuerza_mundial = fuerza[fuerza['equipo'].isin(equipos_encontrados)].copy()
fuerza_mundial = fuerza_mundial.sort_values('idx_ataque', ascending=False).reset_index(drop=True)
fuerza_mundial.index += 1
print(f"\n✓ Tabla actualizada con {len(fuerza_mundial)} equipos")

# BLOQUE 10: Duelo directo Portugal vs Uzbekistan
predecir_partido("Portugal", "Uzbekistan")

# BLOQUE 11: Simulación completa Grupo K
# Portugal, Uzbekistan, Colombia, DR Congo

grupo_k = ["Portugal", "Uzbekistan", "Colombia", "DR Congo"]

# Todos los partidos del grupo (6 duelos)
import itertools

print("PREDICCIONES — GRUPO K")
print("=" * 45)

resultados_grupo = []

for equipo_a, equipo_b in itertools.combinations(grupo_k, 2):
    # Verificar nombres en dataset
    ea = "Czech Republic" if equipo_a == "Czechia" else equipo_a
    eb = "Czech Republic" if equipo_b == "Czechia" else equipo_b

    lam_a, lam_b, prob_a, prob_emp, prob_b = predecir_partido(ea, eb)

    resultados_grupo.append({
        'equipo_a': equipo_a,
        'equipo_b': equipo_b,
        'goles_a':  round(lam_a, 2),
        'goles_b':  round(lam_b, 2),
        'prob_a':   round(prob_a * 100, 1),
        'prob_emp': round(prob_emp * 100, 1),
        'prob_b':   round(prob_b * 100, 1)
    })

    # BLOQUE 12: Tabla de posiciones simulada (puntos esperados)
# Calculamos puntos esperados por probabilidad

print("\n\nTABLA DE POSICIONES ESPERADA — GRUPO K")
print("=" * 50)

puntos = {eq: 0.0 for eq in grupo_k}
goles_favor    = {eq: 0.0 for eq in grupo_k}
goles_contra   = {eq: 0.0 for eq in grupo_k}

for r in resultados_grupo:
    ea, eb = r['equipo_a'], r['equipo_b']

    # Puntos esperados según probabilidad
    puntos[ea] += r['prob_a']/100 * 3 + r['prob_emp']/100 * 1
    puntos[eb] += r['prob_b']/100 * 3 + r['prob_emp']/100 * 1

    # Goles esperados acumulados
    goles_favor[ea]  += r['goles_a']
    goles_favor[eb]  += r['goles_b']
    goles_contra[ea] += r['goles_b']
    goles_contra[eb] += r['goles_a']

# Ordenar por puntos
tabla = sorted(puntos.items(), key=lambda x: x[1], reverse=True)

print(f"{'Pos':<4} {'Equipo':<20} {'Pts esp.':>9} {'GF':>6} {'GC':>6} {'DG':>6}")
print("─" * 50)
for i, (eq, pts) in enumerate(tabla, 1):
    gf = goles_favor[eq]
    gc = goles_contra[eq]
    print(f"{i:<4} {eq:<20} {pts:>9.2f} {gf:>6.2f} {gc:>6.2f} {gf-gc:>+6.2f}")

print("\n✓ Los 2 primeros clasifican a la fase de grupos (Ronda de 32)")

print("GOLES ESPERADOS — GRUPO K")
print("Cada celda = goles que marca el equipo de la FILA contra el de la COLUMNA")
print()

# Encabezado
print(f"{'':20}", end="")
for eq in grupo_k:
    print(f"{eq[:12]:>14}", end="")
print()
print("─" * (20 + 14 * len(grupo_k)))

# Filas
for equipo_a in grupo_k:
    print(f"{equipo_a:<20}", end="")
    for equipo_b in grupo_k:
        if equipo_a == equipo_b:
            print(f"{'---':>14}", end="")
        else:
            fila_a = fuerza[fuerza['equipo'] == equipo_a]
            fila_b = fuerza[fuerza['equipo'] == equipo_b]

            idx_atq_a = fila_a['idx_ataque'].values[0]
            idx_def_b = fila_b['idx_defensa'].values[0]

            lambda_a = idx_atq_a * idx_def_b * promedio_goles
            print(f"{lambda_a:>14.2f}", end="")
    print()

print()
print("Ejemplo: Portugal vs Uzbekistan → goles esperados de Portugal en esa celda")

# BLOQUE 14: Marcador más probable — partido a partido Grupo K

from scipy.stats import poisson
import numpy as np
import itertools

print("MARCADOR MÁS PROBABLE — GRUPO K")
print("=" * 55)

for equipo_a, equipo_b in itertools.combinations(grupo_k, 2):
    fila_a = fuerza[fuerza['equipo'] == equipo_a]
    fila_b = fuerza[fuerza['equipo'] == equipo_b]

    lam_a = fila_a['idx_ataque'].values[0] * fila_b['idx_defensa'].values[0] * promedio_goles
    lam_b = fila_b['idx_ataque'].values[0] * fila_a['idx_defensa'].values[0] * promedio_goles

    # Buscar el marcador más probable (0-0 hasta 5-5)
    max_prob = 0
    marcador = (0, 0)
    for g_a in range(6):
        for g_b in range(6):
            prob = poisson.pmf(g_a, lam_a) * poisson.pmf(g_b, lam_b)
            if prob > max_prob:
                max_prob = prob
                marcador = (g_a, g_b)

    # Top 3 marcadores más probables
    marcadores = []
    for g_a in range(6):
        for g_b in range(6):
            prob = poisson.pmf(g_a, lam_a) * poisson.pmf(g_b, lam_b)
            marcadores.append((g_a, g_b, prob))

    marcadores.sort(key=lambda x: x[2], reverse=True)

    print(f"\n  {equipo_a}  vs  {equipo_b}")
    print(f"  Goles esperados : {lam_a:.2f} — {lam_b:.2f}")
    print(f"  Top 3 marcadores más probables:")
    for g_a, g_b, prob in marcadores[:3]:
        barra = "◆" if (g_a, g_b) == (marcadores[0][0], marcadores[0][1]) else "◇"
        print(f"    {barra} {g_a} - {g_b}  ({prob*100:.1f}%)")



Total de partidos descargados: 49477

Columnas disponibles:
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Primeros 3 registros:
Partidos de Mundial y Clasificatorias desde 1990: 7601

Torneos incluidos:
tournament
FIFA World Cup qualification    6977
FIFA World Cup                   624
Name: count, dtype: int64
Total de registros por equipo: 15202

Ejemplo — partidos de Brasil:
Promedio global de goles por partido: 1.425

Top 10 equipos más goleadores:
     equipo  partidos_jugados   ataque  defensa  idx_ataque  idx_defensa
New Zealand                51 2.647059 0.862745    1.857073     0.605268
    Germany               119 2.605042 0.789916    1.827596     0.554174
       Fiji                26 2.576923 1.730769    1.807869     1.214240
  Australia               135 2.481481 0.777778    1.740911     0.545659
Netherlands               127 2.480315 0.724409    1.740092     0.508217
      Japan               146 2.383562 0.6